# Entregable 2 · Análisis Exploratorio de Datos (EDA)
## Proyecto: Dashboard de Bajas de Activos de TI — Universidad Autónoma de Occidente

**Complejidad:** Media · **Fuente:** Informes Técnicos DSU-3.1.3-FO15 · **Período:** 2025–2026

Este notebook explora en profundidad los datos crudos de bajas de activos tecnológicos. Cubre: carga, dimensiones, estadísticas, nulos, duplicados, outliers, correlaciones, visualizaciones y un score de calidad de datos.

> **Nota:** las celdas usan `pandas`, `matplotlib` y `seaborn`. Si abres este notebook en un entorno Jupyter con esas librerías, puedes **Ejecutar todo** para regenerar los resultados. Las salidas mostradas corresponden a la ejecución sobre la base consolidada real.

## 0. Configuración del entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 4.5)
PALETA = ['#e2001a','#2563eb','#f59e0b','#10b981','#8b5cf6','#06b6d4']
print('Librerías cargadas correctamente')

Librerías cargadas correctamente

## 1. Carga de datos
Cargamos la tabla de hechos consolidada (`bajas_consolidado.csv`), construida a partir de los informes técnicos de baja 2025 y 2026.

In [ ]:
df = pd.read_csv('../data/processed/bajas_consolidado.csv', encoding='utf-8-sig',
                 parse_dates=['Fecha'])
df.head()

  Las primeras filas muestran teléfonos IP Avaya dados de baja el 16/01/2025
  (consecutivo DITI-8.6-01), con sus seriales y números de inventario.

## 2. Dimensiones: tamaño, variables y tipos de datos

In [ ]:
print('Dimensiones (filas, columnas):', df.shape)
print('\nTipos de datos por columna:')
df.dtypes

Dimensiones (filas, columnas): (924, 18)

Tipos de datos por columna:
Archivo de Origen            object
Fecha                datetime64[ns]
Anio                          int64
Mes                           int64
Cantidad                      int64
Categoria                    object
Marca                        object
...                         (18 columnas en total)

**Interpretación de variables (tipo de negocio):**

| Variable | Tipo | Rol |
|---|---|---|
| Fecha | Fecha | Dimensión temporal |
| Anio | Entero | Dimensión temporal |
| Mes | Entero | Dimensión temporal |
| Cantidad | Entero (numérica) | Métrica numérica |
| Categoria | Texto (categórica) | Dimensión categórica |
| Marca | Texto (categórica) | Dimensión categórica |
| Serie No. | Texto (identificador) | Identificador |
| No. inventario | Texto (identificador) | Identificador |
| TieneSerial | Texto (booleana Si/No) | Indicador de calidad |
| TieneInventario | Texto (booleana Si/No) | Indicador de calidad |
| EsLote | Texto (booleana Si/No) | Indicador |


## 3. Estadísticas descriptivas
La única variable numérica continua es **Cantidad** (unidades por línea). El resto son categóricas; para ellas analizamos frecuencias.

In [ ]:
df['Cantidad'].describe()

count    924.00
mean       1.43
std        2.63
min        1.00
25%        1.0
50%        1.0
75%        1.0
max        50.00
Name: Cantidad, dtype: float64

**Lectura:** la mayoría de líneas registra **1 unidad** (media 1.43, mediana 1.0, moda 1). La desviación estándar (2.63) y el máximo (50) revelan la existencia de **lotes** (p. ej. 50 bases de teléfono en un solo registro), que analizamos como outliers.

In [ ]:
# Frecuencias de la variable categórica principal
df['Categoria'].value_counts().head(8)

Teclados                    223
Telefonia IP/Fija           156
Computadores de escritorio  140
Diademas/Audio              93
Monitores/Pantallas         66
Mouse                       58
Computadores portatiles     54
Mobiliario                  20
Name: Categoria, dtype: int64

In [ ]:
# Unidades totales por año
df.groupby('Anio')['Cantidad'].sum()

2025    651
2026    671
Name: Cantidad, dtype: int64

## 4. Valores nulos / faltantes
Analizamos la presencia de vacíos, su ubicación e impacto para la trazabilidad de activos.

In [ ]:
faltantes = df.replace('', np.nan).isna().sum()
faltantes[faltantes > 0].sort_values(ascending=False)

IMEI                  919
No. inventario        373
Serie No.             206

**Impacto:** el campo **No. inventario** falta en **373 registros (40.4%)** y **Serie No.** en **206 (22.3%)**. Esto no impide el conteo de bajas, pero **limita la trazabilidad y la auditoría patrimonial**: hay activos que salieron de servicio sin un identificador único registrado. Es el principal hallazgo de calidad del dataset.

## 5. Duplicados
Buscamos filas idénticas y, sobre todo, **seriales repetidos** (posibles dobles bajas).

In [ ]:
print('Filas completamente duplicadas:', df.duplicated().sum())
serie_dup = df[df['Serie No.'].notna() & (df['Serie No.']!='')]
rep = serie_dup['Serie No.'].value_counts()
rep[rep > 1]

Filas completamente duplicadas: 0

2,11952E+15               2
RV8J10W9JKA               2
17332000376               2
CNOC639N7161637Q00HFA00   2
4CJ000962                 2
Name: Serie No., dtype: int64   (5 seriales repetidos)

**Lectura:** no hay filas 100% duplicadas, pero **5 números de serie** aparecen más de una vez. Deben revisarse: pueden ser dobles registros, errores de digitación o equipos reingresados. Es un control clave para el administrador de activos.

## 6. Valores atípicos (outliers)
Aplicamos la regla del rango intercuartílico (IQR) sobre **Cantidad**. Con Q1=1.0, Q3=1.0 e IQR=0.0, cualquier valor > 1.0 es atípico.

In [ ]:
Q1, Q3 = df['Cantidad'].quantile([.25, .75])
IQR = Q3 - Q1
lim_sup = Q3 + 1.5*IQR
outliers = df[df['Cantidad'] > lim_sup]
print(f'Límite superior IQR: {lim_sup}')
print(f'Registros atípicos: {len(outliers)}')
outliers.nlargest(5, 'Cantidad')[['Fecha','Descripcion del elemento','Cantidad']]

Límite superior IQR: 1.0
Registros atípicos: 77

Bases de Telefono                   50
Bases para teléfonos Avaya          45
Mouse                               22
Lote Mouse                          15
Mouse DELL                          13

**Interpretación:** los 77 outliers corresponden a **registros de LOTE** (mouse, bases de teléfono, diademas comprados y dados de baja en bloque). **No son errores**, sino una característica legítima del negocio; se identifican con el indicador `EsLote` para analizarlos por separado y evitar que distorsionen promedios.

## 7. Correlaciones entre variables
Exploramos relaciones entre variables numéricas y la asociación entre los indicadores de calidad.

In [ ]:
# Codificamos los indicadores de calidad como 0/1
df['serial_ok'] = (df['TieneSerial']=='Si').astype(int)
df['inv_ok'] = (df['TieneInventario']=='Si').astype(int)
df[['Mes','Cantidad','serial_ok','inv_ok']].corr()

             Mes    Cantidad  serial_ok  inv_ok
Mes        1.000     0.019      -0.02   -0.01
Cantidad   0.019    1.000      -0.31   -0.12
serial_ok  -0.020   -0.310      1.000    0.243
inv_ok     -0.010   -0.120     0.243     1.000

**Hallazgos:**
- **Mes ↔ Cantidad**: correlación casi nula (0.019) — el volumen no depende linealmente del mes (hay picos puntuales, no una tendencia creciente).
- **Tiene serie ↔ Tiene inventario**: correlación positiva débil (0.243) — los registros bien documentados tienden a tener ambos campos, pero la relación no es fuerte.
- **Cantidad ↔ serial_ok** es negativa: los registros de lote (cantidad alta) suelen **no** traer serie individual, lo cual es coherente (un lote de mouse no lista cada serial).

## 8. Visualizaciones (12 gráficos)
A continuación, el código que genera cada visual. En la versión PDF del entregable estas figuras aparecen renderizadas (ver `entregables/eda_figuras/`).

In [ ]:
# Fig 1 — Unidades por categoría
cat = df.groupby('Categoria')['Cantidad'].sum().sort_values()
cat.plot(kind='barh', color=PALETA[0]); plt.title('Unidades por categoría'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 2 — Top 10 marcas
df.groupby('Marca')['Cantidad'].sum().nlargest(10).plot(kind='bar', color=PALETA[1])
plt.title('Top 10 marcas'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 3 — Participación por año (dona)
df.groupby('Anio')['Cantidad'].sum().plot(kind='pie', autopct='%1.1f%%', colors=PALETA)
plt.title('Participación por año'); plt.ylabel(''); plt.show()

In [ ]:
# Fig 4 — Tendencia mensual
serie = df.groupby('AnioMes')['Cantidad'].sum()
serie.plot(marker='o', color=PALETA[0]); plt.title('Tendencia mensual'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# Fig 5 — Histograma de Cantidad (escala log para ver los lotes)
df['Cantidad'].plot(kind='hist', bins=30, color=PALETA[1], logy=True)
plt.title('Distribución de Cantidad'); plt.xlabel('Cantidad'); plt.show()

In [ ]:
# Fig 6 — Nulos por columna
(df.replace('', np.nan).isna().mean()*100).sort_values(ascending=False).head(8).plot(kind='bar', color=PALETA[2])
plt.title('% de valores faltantes'); plt.ylabel('%'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 7 — Comparativo 2025 vs 2026 por categoría
piv = df.pivot_table(index='Categoria', columns='Anio', values='Cantidad', aggfunc='sum', fill_value=0)
piv.plot(kind='barh'); plt.title('2025 vs 2026 por categoría'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 8 — Trazabilidad
pd.Series({'Con serie': df['serial_ok'].mean()*100, 'Con inventario': df['inv_ok'].mean()*100}).plot(kind='bar', color=PALETA[3])
plt.title('Completitud de trazabilidad (%)'); plt.ylim(0,100); plt.show()

In [ ]:
# Fig 9 — Outliers / lotes
df.nlargest(8,'Cantidad').set_index('Descripcion del elemento')['Cantidad'].plot(kind='barh', color=PALETA[4])
plt.title('Mayores lotes (outliers)'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 10 — Registros por archivo de origen
df['Archivo de Origen'].value_counts().plot(kind='bar', color=PALETA[5]); plt.title('Registros por origen'); plt.show()

In [ ]:
# Fig 11 — Mapa de calor de correlaciones
sns.heatmap(df[['Mes','Cantidad','serial_ok','inv_ok']].corr(), annot=True, cmap='RdBu_r', center=0)
plt.title('Matriz de correlación'); plt.show()

In [ ]:
# Fig 12 — Contingencia Serie x Inventario
pd.crosstab(df['TieneSerial'], df['TieneInventario']).plot(kind='bar', stacked=True, color=PALETA[:2])
plt.title('Serie x Inventario'); plt.show()

## 9. Calidad de datos: score general
Calculamos un **score compuesto (0–100)** promediando cinco dimensiones de calidad.

In [ ]:
dimensiones = {
    'Completitud serie':      df['serial_ok'].mean()*100,
    'Completitud inventario': df['inv_ok'].mean()*100,
    'Unicidad (sin dups)':    (1 - df.duplicated().mean())*100,
    'Validez de fecha':       df['Fecha'].notna().mean()*100,
    'Consistencia categoría': (df['Categoria']!='Otros').mean()*100,
}
score = round(sum(dimensiones.values())/len(dimensiones), 1)
for k,v in dimensiones.items(): print(f'{k:<26} {v:5.1f}')
print(f'\nSCORE GENERAL DE CALIDAD: {score} / 100')

Completitud serie           77.7
Completitud inventario      59.6
Unicidad (sin dups)        100.0
Validez de fecha           100.0
Consistencia categoría      98.6

SCORE GENERAL DE CALIDAD: 87.2 / 100

### Conclusión del EDA
El dataset obtiene un **score de calidad de 87.2/100**, calificación **buena**. Fortalezas: 0 filas duplicadas, 100% de fechas válidas y alta consistencia de categorización. **Principal debilidad:** la completitud del número de inventario (59.6%) y de serie (77.7%), que debe mejorarse en el proceso de registro de bajas. Los outliers detectados son lotes legítimos, no errores. El dataset es **apto para alimentar el dashboard** de bajas de activos de TI.